# Modelo de Identificação de Fraudes

- Informações da base:
    - 'Time' (tempo): contém os segundos decorridos entre cada transação e a primeira transação no conjunto de dados. 
    - 'Amount' (valor): é o valor da transação 
    - 'Class' (classe): é a variável de resposta e assume valor 1 em caso de fraude e 0 caso contrário.
    - "Infelizmente, devido a questões de confidencialidade, não podemos fornecer os recursos originais e mais informações básicas sobre os dados. Características V1, V2, … V28 são os principais componentes obtidos com PCA"

In [ ]:
# Configuração do ambiente

# Garante que o notebook está na raiz do projeto
%cd .. 

# Verifica o diretório atual (Linux/Mac)
# !pwd  

# Verifica o diretório atual (Windows)
!cd 

In [ ]:
# Importações necessárias 

import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime as dt
import xgboost as xgb

from scipy.stats import kstest, normaltest, anderson

from sklearn.preprocessing import PowerTransformer, MinMaxScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree, metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids, NearMiss 
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTEENN

from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,
)

import warnings
from src.config.paths import RAW_DATA_PATH, PROCESSED_DATA_PATH
from src.config.auxiliares_ml import downcast_dataframe, testar_modelos_com_undersampling

warnings.filterwarnings('ignore')
sns.set_theme(palette="bright")

In [ ]:
# Importando a base

transacoes = pd.read_parquet(PROCESSED_DATA_PATH)

In [ ]:
# Visualizando estatísticas descritivas da base 

transacoes.describe()

### Pré-processamento: 

- Utilizaremos o **PowerTransformer** devido ao grande número de outliers, **xceto em 'Time'**. <br>
- Em **'Time'** utilizaremos o **MinMaxScaler**. <br>
- Vamos utilizar o **RandomUnderSampler** para balancear nossa base.

In [ ]:
# Criando os escaladores

power_transformer = PowerTransformer(method='yeo-johnson')  # Funciona para valores positivos e negativos
minmax_scaler = MinMaxScaler()

# Aplicando a normalização nas colunas específicas

transacoes['Amount'] = power_transformer.fit_transform(transacoes[['Amount']])
transacoes['Time'] = minmax_scaler.fit_transform(transacoes[['Time']])


transacoes.head(2)

In [ ]:
# Separando X e y

X = transacoes.drop(columns='Class')
y = transacoes['Class']

### Fazendo Cross-validation com StratifiedKFold e utilizando o RandomUnderSampler

In [ ]:
# Configuração do StratifiedKFold

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

# Criando o RandomUnderSampler

rus = RandomUnderSampler(random_state=0)

# Lista para armazenar os conjuntos de treino e teste de cada dobra

folds_data = []

for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Aplicando o balanceamento com RandomUnderSampler nos dados de treino
    
    X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

    # Armazenando os dados da dobra
    
    folds_data.append((X_train_bal, y_train_bal, X_test, y_test))

print("Cross-validation com balanceamento concluído. Agora podemos testar os modelos.")

### XGBoost 

In [ ]:
# Criando o modelo XGBoost

modelo_xgb = xgb.XGBClassifier(
    max_depth=9,
    n_estimators=200,
    learning_rate=0.3,
    random_state=0,
    scale_pos_weight=(y.value_counts()[0] / y.value_counts()[1])  # Lida com desbalanceamento
)

# Loop pelas dobras do cross-validation

for i, (X_train_bal, y_train_bal, X_test, y_test) in enumerate(folds_data):
    modelo_xgb.fit(X_train_bal, y_train_bal)  # Treina no conjunto balanceado
    y_pred_xgb = modelo_xgb.predict(X_test)  # Testa no conjunto original
    
    # Calculando métricas
    
    acc = accuracy_score(y_test, y_pred_xgb)
    prec = precision_score(y_test, y_pred_xgb, zero_division=0)  # zero_division evita erros se não houver positivos
    rec = recall_score(y_test, y_pred_xgb)

    # Exibindo os resultados
    
    print(f"Dobra {i+1}:")
    print(f"  Acurácia:  {acc:.4f}")
    print(f"  Precisão:  {prec:.4f}")
    print(f"  Recall:    {rec:.4f}\n")

### Random Forest

In [ ]:
# Utilizando o Random Forest

clf_RF = RandomForestClassifier(
    max_depth=7,  
    random_state=0,
    criterion='gini',
    n_estimators=100,  
    class_weight='balanced_subsample' 
).fit(X_train, y_train)

# Loop pelas dobras do cross-validation

for i, (X_train_bal, y_train_bal, X_test, y_test) in enumerate(folds_data):
    clf_RF.fit(X_train_bal, y_train_bal)  # Treina no conjunto balanceado
    y_pred_RF = clf_RF.predict(X_test)  # Testa no conjunto original
    
    # Calculando métricas
    
    acc = accuracy_score(y_test, y_pred_RF)
    prec = precision_score(y_test, y_pred_RF, zero_division=0)  # zero_division evita erros se não houver positivos
    rec = recall_score(y_test, y_pred_RF)

    # Exibindo os resultados
    
    print(f"Dobra {i+1}:")
    print(f"  Acurácia:  {acc:.4f}")
    print(f"  Precisão:  {prec:.4f}")
    print(f"  Recall:    {rec:.4f}\n")

### OBSERVAÇÃO:

**Utilizamos apenas os dois algoritmos que se saíram melhor até aqui:**.
- XGBoost
- Random Forest

### NOTA:

Observamos que sempre que executamos o balanceamento da base, os resultados não são satisfatórios.